<a href="https://colab.research.google.com/github/chaseshallow/ansci_4040_final_class_project/blob/main/Step02cSam2TrackCompare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 02c · Experiment: track with SAM 2.1 instead of ByteTrack

**Question:** ByteTrack (Step 02b) split one cow into 47 tracks, mostly while she lay down and strained. Does **SAM 2.1's video mode**, which follows the cow's *mask* using a memory of how she looks, keep her as fewer, longer tracks?

**How:** YOLO26n finds the cow at the start of each chunk (2 min), SAM 2.1 Small follows her mask through the chunk, and chunks are joined where the boxes overlap. Every SAM box is compared with YOLO's boxes in the same frame:

- **Agrees**: YOLO sees the cow in the same place
- **NoYoloBox**: YOLO saw nothing, SAM kept following her (this is what we hope for)
- **Disagrees**: YOLO sees a cow elsewhere but not where SAM is; SAM may have **drifted** (calf, floor): check the preview

Default window: **1080 to 2500 s** of the clip (about 03:53 to 04:17 in the video), where ByteTrack broke into dozens of short tracks. Try a short window first (e.g. `EndSeconds = 1200`).

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step02cSam2/`, **overwriting** each run:

- `Comparison.csv`: ByteTrack vs SAM 2 on the same window (tracks, coverage, longest / median track)
- `Tracks.csv` (same columns as 02b, so step 03 could use it), `TrackSummary.csv`
- `PreviewTracks.mp4`: masks + boxes + IDs, label `ok` / `sam` / `??` = Agrees / NoYoloBox / Disagrees

Needs a **T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO/SAM.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/ansci_4040_final_class_project.git"
RepoBranch = "main"
RepoDir = "/content/ansci_4040_final_class_project"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Sam2Track import Sam2TrackWindow
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem
ClipPath = VideoOutputDir / "ReferenceClip.mp4"                       # step 01
CompareTracksPath = VideoOutputDir / "Step02bFull" / "Tracks.csv"     # ByteTrack result to compare with
OutputDir = VideoOutputDir / "Step02cSam2"

StartSeconds = 1080    # seconds into the clip (add 03:35:11 for video time)
EndSeconds = 2500      # try 1200 first for a quick test (~2 min)
TargetFps = 5          # same rate as 02b, so the comparison is fair
ChunkSeconds = 120     # SAM 2 memory grows per frame; restart it from a YOLO box every 2 min
SeedConfidence = 0.4   # YOLO box needed to start following a cow
PreviewSpeed = 4

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"

In [ ]:
# 3. Run: track with SAM 2, then compare with ByteTrack and show the preview.
import pandas as pd
from IPython.display import Video, display

Stats = Sam2TrackWindow(ClipPath, OutputDir, StartSeconds, EndSeconds, TargetFps=TargetFps, ChunkSeconds=ChunkSeconds,
                        SeedConfidence=SeedConfidence, CompareTracksPath=CompareTracksPath, PreviewSpeed=PreviewSpeed)

display(pd.read_csv(OutputDir / "Comparison.csv"))
display(pd.read_csv(OutputDir / "TrackSummary.csv"))
PreviewPath = OutputDir / "PreviewTracks.mp4"
PreviewMb = PreviewPath.stat().st_size / 2**20
print(f"Preview: {PreviewPath} ({PreviewMb:.0f} MB)")
if PreviewMb < 50:   # bigger files are too heavy to play inside the notebook; open them from Drive
    display(Video(str(PreviewPath), embed=True, width=960))

## About this step

**Models:** **YOLO26n** (detection, only to start and check) + **SAM 2.1 Small in video mode** (Ultralytics `SAM2VideoPredictor`, `sam2.1_s.pt`) for following the cow. ByteTrack is not used.

**How to judge the result:** in `Comparison.csv`, fewer tracks, a higher `FramesWithCowShare` and longer tracks are better, **but only if** the preview shows the masks staying on the cow. A high `Disagrees` share, or masks that slide onto the calf or the floor, means SAM is drifting.

**Decision:** if SAM 2 wins, its `Tracks.csv` can replace 02b's as the input to step 03. If not, we keep ByteTrack.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.